In [ ]:
from pyspark.sql import SparkSession
from openai import OpenAI
import os

# 1. Initialize Spark
print("Starting Spark...")
spark = SparkSession.builder \
    .appName("ModelAgent") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# 2. Initialize OpenAI (Paste in your own API key)
client = OpenAI(api_key='')

# 3. Load the unzipped parquet folder into memory
print("Loading engineered data...")
engineered_df = spark.read.parquet("engineered_df.parquet")

# 4. Recreate the final state from your Feature Engineering Agent
# I pulled these exact columns and stats from your previous notebook logs!
fe_final_state = {
    "engineered_df_path": "engineered_df.parquet",
    "feature_cols": [
        'is_rush_hour', 'Weather_Condition_idx', 'Wind_Direction_ohe', 
        'State_ohe', 'Sunrise_Sunset_ohe', 'Start_Time_Hour', 
        'Start_Time_is_Weekend', 'Visibility(mi)_bin', 'Temperature(F)_bin', 
        'Wind_Speed(mph)_bin', 'Distance(mi)', 'Duration_Minutes', 
        'Distance(mi)_ratio_Duration_Minutes'
    ],
    "post_cleaning_profile": {
        "target_analysis": {
            "column": "Severity",
            "class_distribution": {"4": 124888, "1": 61934, "3": 613922, "2": 4469929},
            "imbalance_ratio": 72.17,
            "imbalance_classification": "severe_imbalance"
        }
    }
}
print("Data and state loaded successfully! Ready to run the Model Agent.")

Starting Spark...


26/04/11 22:32:51 WARN Utils: Your hostname, Ericas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.31.10.49 instead (on interface en0)
26/04/11 22:32:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/11 22:32:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Loading engineered data...


Data and state loaded successfully! Ready to run the Model Agent.


In [2]:
import json
import pandas as pd
from typing import List, Optional, Dict, Any, Literal, TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END
import numpy as np
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
from pyspark.sql.functions import col, when

# PySpark ML Imports
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression, GBTClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col, when

# Imblearn Imports
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek

## schemas and state def

In [3]:
class ModelParams(BaseModel):
    maxDepth: List[int] = Field(description="Tree depths to test (e.g., [5, 10]). Return empty [] if not applicable to the model.")
    regParam: List[float] = Field(description="Regularization values (e.g., [0.01, 0.1]). Return empty [] if not applicable.")

class ModelSuggestion(BaseModel):
    target_type: Literal["binary", "multi-class"]
    rationale_target: str
    primary_model: Literal["LogisticRegression", "RandomForest", "GBTClassifier"]
    secondary_model: Literal["LogisticRegression", "RandomForest", "GBTClassifier"]
    primary_model_grid: ModelParams
    secondary_model_grid: ModelParams

class AgentState(TypedDict):
    df: object
    engineered_df_path: str
    feature_cols: List[str]
    post_cleaning_profile: dict
    
    # Model Agent State Keys
    model_suggestion: Optional[dict]
    human_approved_target: Optional[str]
    human_approved_models: Optional[List[str]]
    evaluation_metrics: Optional[Dict[str, float]]
    trained_models: Optional[Dict[str, Any]]

## langraph

In [4]:
def model_suggestion_agent(state: AgentState) -> AgentState:
    print("=" * 60)
    print(">> MODEL AGENT: Analyzing data for 2-Model Selection")
    print("=" * 60)
    
    profile = state.get("post_cleaning_profile", {})
    target_analysis = profile.get("target_analysis", {})
    
    system_prompt = """You are a senior PySpark ML architect.
    Review the target distribution and recommend whether to use binary or multi-class classification.

    CRITICAL OBJECTIVE: Strongly push for 'binary' classification to avoid minority class collapse on imbalanced datasets. Justify this.
    
    Recommend TWO PySpark MLlib models to compare. Provide realistic hyperparameter grids for BOTH models using the strict schema. 
    If a parameter does not apply (like maxDepth for LogisticRegression), return an empty list []."""
    
    user_prompt = f"Target Column Analysis: {json.dumps(target_analysis)}\n\nProduce a ModelSuggestion."
    
    # Using your initialized OpenAI client
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format=ModelSuggestion,
    )
    
    suggestion = response.choices[0].message.parsed
    print(f"Agent Suggests Target: {suggestion.target_type}")
    print(f"Primary Model: {suggestion.primary_model}")
    print(f"Secondary Model: {suggestion.secondary_model}")
    
    return {**state, "model_suggestion": suggestion.model_dump()}


def human_approval_node(state: AgentState) -> AgentState:
    suggestion = state["model_suggestion"]
    
    print("\n" + "=" * 60)
    print(">> HUMAN-IN-THE-LOOP INTERRUPT")
    print("=" * 60)
    print(f"Agent suggested a {suggestion['target_type']} approach.")
    print(f"Models to compare: 1) {suggestion['primary_model']}  2) {suggestion['secondary_model']}")
    
    # Target Type Approval
    print("\n--- TARGET SELECTION ---")
    user_target = input(f"Type 'binary' or 'multi-class' (or press Enter to accept agent's '{suggestion['target_type']}'): ").strip()
    final_target = user_target if user_target else suggestion['target_type']
    
    # Model Approval
    print("\n--- MODEL SELECTION ---")
    print("Type two models separated by a comma, or press Enter to accept the agent's models.")
    print("(Options: LogisticRegression, RandomForest, GBTClassifier)")
    user_models = input("Input: ").strip()
    
    if user_models:
        final_models = [m.strip() for m in user_models.split(",")]
    else:
        final_models = [suggestion['primary_model'], suggestion['secondary_model']]
    
    print(f"\nProceeding with {final_target} classification comparing: {final_models}")
    
    return {
        **state, 
        "human_approved_target": final_target,
        "human_approved_models": final_models
    }

def execute_model_node(state: AgentState) -> AgentState:
    print("\n" + "=" * 60)
    print(">> EXECUTING 2-MODEL PIPELINE (SEQUENTIAL)")
    print("=" * 60)
    
    df = state["df"] 
    raw_feature_cols = state["feature_cols"]
    models_to_run = state["human_approved_models"]

    actual_columns = df.columns
    feature_cols = [c for c in raw_feature_cols if c in actual_columns]
    
    missing_cols = [c for c in raw_feature_cols if c not in actual_columns]
    if missing_cols:
        print(f"WARNING: The following expected features are missing from the dataset and will be skipped: {missing_cols}")
    
    if state["human_approved_target"] == "binary":
        print("0. Converting multi-class 'Severity' to 'Severity_Binary' (Severity >= 3 becomes 1, else 0)...")
        df = df.withColumn("Severity_Binary", when(col("Severity") >= 3, 1).otherwise(0))
        target_col = "Severity_Binary"
    else:
        target_col = "Severity"

    print("0.5. Assembling PySpark Vector columns...")
    if "features" in df.columns:
        df = df.drop("features")
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
    df_assembled = assembler.transform(df).select("features", target_col)
    
    # ---------------------------------------------------------
    # PART 1: Train-Test Split
    # ---------------------------------------------------------
    print("1. Performing 80/20 Train-Test Split...")
    train_df, test_df = df_assembled.randomSplit([0.8, 0.2], seed=42)

    # ---------------------------------------------------------
    # PART 2: Safe Resampling (Training data)
    # ---------------------------------------------------------
    print("2. Downsampling majority class natively in PySpark (Train Set)...")
    fractions = {0: 0.05, 1: 0.20}
    sampled_train_df = train_df.stat.sampleBy(target_col, fractions, seed=42)
    
    print("3. Converting Train Set to Pandas & extracting dense arrays...")
    pdf_train = sampled_train_df.toPandas()
    
    # Extract PySpark Vectors into a flat 2D NumPy array for SMOTE
    X = np.stack(pdf_train["features"].apply(lambda v: v.toArray()))
    y = pdf_train[target_col].values
    
    print("4. Applying SMOTETomek (This may take a moment)...")
    smt = SMOTETomek(smote=SMOTE(random_state=42), tomek=TomekLinks())
    X_res, y_res = smt.fit_resample(X, y)
    
    print("5. Converting balanced data back to PySpark Vectors...")
    # Map the numpy arrays back into PySpark DenseVectors
    rdd = spark.sparkContext.parallelize(zip(X_res, y_res)).map(lambda x: (Vectors.dense(x[0]), int(x[1])))
    schema = StructType([
        StructField("features", VectorUDT(), True),
        StructField(target_col, IntegerType(), True)
    ])
    balanced_train_df = spark.createDataFrame(rdd, schema)

    # We need the evaluator outside the loop
    evaluator = BinaryClassificationEvaluator(labelCol=target_col) if state["human_approved_target"] == "binary" else MulticlassClassificationEvaluator(labelCol=target_col)
    
    evaluation_results = {}
    trained_models_dict = {}
    
    # ---------------------------------------------------------
    # PART 3: Sequential Model Training Loop
    # ---------------------------------------------------------
    for model_choice in models_to_run:
        print(f"\n--- Training {model_choice} ---")
                
        # Instantiate correct classifier
        if model_choice == "RandomForest":
            classifier = RandomForestClassifier(featuresCol="features", labelCol=target_col)
        elif model_choice == "LogisticRegression":
            classifier = LogisticRegression(featuresCol="features", labelCol=target_col)
        else:
            classifier = GBTClassifier(featuresCol="features", labelCol=target_col)
            
        pipeline = Pipeline(stages=[classifier])
        
        # Pull the correct strict schema grid
        if model_choice == state["model_suggestion"]["primary_model"]:
            grid_params = state["model_suggestion"]["primary_model_grid"]
        else:
            grid_params = state["model_suggestion"]["secondary_model_grid"]
            
        grid_builder = ParamGridBuilder()
        if len(grid_params["maxDepth"]) > 0 and hasattr(classifier, "maxDepth"):
            grid_builder.addGrid(classifier.maxDepth, [int(x) for x in grid_params["maxDepth"]])
        if len(grid_params["regParam"]) > 0 and hasattr(classifier, "regParam"):
            grid_builder.addGrid(classifier.regParam, grid_params["regParam"])
            
        paramGrid = grid_builder.build()
        
        cv = CrossValidator(estimator=pipeline, 
                            estimatorParamMaps=paramGrid, 
                            evaluator=evaluator, 
                            numFolds=3, 
                            seed=42)
        
        print(f"Fitting CrossValidator on Balanced Train Set for {model_choice}...")
        cvModel = cv.fit(balanced_train_df)
        
        print(f"Evaluating {model_choice} on Unseen Test Set...")
        predictions = cvModel.transform(test_df)
        test_metric = evaluator.evaluate(predictions)
        
        evaluation_results[model_choice] = test_metric
        trained_models_dict[model_choice] = cvModel.bestModel
        
        print(f"{model_choice} Final Test Metric: {test_metric:.4f}")

    print("\n--- Both Models Trained Successfully! ---")
    
    return {
        **state, 
        "evaluation_metrics": evaluation_results,
        "trained_models": trained_models_dict
    }

## graph compilation

In [ ]:
def build_model_subgraph():
    workflow = StateGraph(AgentState)
    workflow.add_node("model_agent", model_suggestion_agent)
    workflow.add_node("human_approval", human_approval_node)
    workflow.add_node("execute_model", execute_model_node)
    
    workflow.set_entry_point("model_agent")
    workflow.add_edge("model_agent", "human_approval")
    workflow.add_edge("human_approval", "execute_model")
    workflow.add_edge("execute_model", END)
    return workflow.compile()

## build pipeline

In [6]:
model_pipeline = build_model_subgraph()

In [8]:
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# 1. Bridge the state from Feature Engineering to the Model Agent
# Assuming your feature engineering output was saved in a variable called 'fe_final_state'
# and the active Spark DataFrame is 'engineered_df'

initial_model_state = {
    "df": engineered_df, # The dataframe returned by your FE step
    "engineered_df_path": fe_final_state.get("engineered_df_path", ""),
    "feature_cols": fe_final_state["feature_cols"], 
    "post_cleaning_profile": fe_final_state.get("post_cleaning_profile", {}),
    
    # Initialize empty keys for the model agent to fill
    "model_suggestion": None,
    "human_approved_target": None,
    "human_approved_models": None,
    "evaluation_metrics": None,
    "trained_models": None
}

print("Starting the Model Selection & Execution Phase...")

# 2. Run the pipeline!
# This will hit the 'human_approval_node' and pause for your input in the output console.
final_model_state = model_pipeline.invoke(initial_model_state)

# 3. View the results
print("\n=== FINAL EVALUATION METRICS ===")
for model_name, score in final_model_state["evaluation_metrics"].items():
    print(f"{model_name}: {score:.4f}")

Starting the Model Selection & Execution Phase...
>> MODEL AGENT: Analyzing data for 2-Model Selection
Agent Suggests Target: binary
Primary Model: LogisticRegression
Secondary Model: RandomForest

>> HUMAN-IN-THE-LOOP INTERRUPT
Agent suggested a binary approach.
Models to compare: 1) LogisticRegression  2) RandomForest

--- TARGET SELECTION ---


Type 'binary' or 'multi-class' (or press Enter to accept agent's 'binary'):  



--- MODEL SELECTION ---
Type two models separated by a comma, or press Enter to accept the agent's models.
(Options: LogisticRegression, RandomForest, GBTClassifier)


Input:  



Proceeding with binary classification comparing: ['LogisticRegression', 'RandomForest']

>> EXECUTING 2-MODEL PIPELINE (SEQUENTIAL)
0. Converting multi-class 'Severity' to 'Severity_Binary' (Severity >= 3 becomes 1, else 0)...
0.5. Assembling PySpark Vector columns...
1. Performing 80/20 Train-Test Split...
2. Downsampling majority class natively in PySpark (Train Set)...
3. Converting Train Set to Pandas & extracting dense arrays...


4. Applying SMOTETomek (This may take a moment)...
5. Converting balanced data back to PySpark Vectors...

--- Training LogisticRegression ---
Fitting CrossValidator on Balanced Train Set for LogisticRegression...


Evaluating LogisticRegression on Unseen Test Set...


LogisticRegression Final Test Metric: 0.6687

--- Training RandomForest ---
Fitting CrossValidator on Balanced Train Set for RandomForest...


Evaluating RandomForest on Unseen Test Set...


RandomForest Final Test Metric: 0.7934

--- Both Models Trained Successfully! ---

=== FINAL EVALUATION METRICS ===
LogisticRegression: 0.6687
RandomForest: 0.7934


In [10]:
spark.stop()